# Creating Networks from Data

This notebook shows several examples of constructing networks from raw datasets in various formats. We will use *Pandas* to load and represent the original data, and *NetworkX* to create the networks. This approach demonstrates how real-world data can be transformed into network representations for analysis.

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## Creating a Network from CSV Data

A common way of storing many types of data is to use *Comma-Separated Values (CSV)* format. As an example, we will read a CSV file containing route data covering the Irish Rail network sourced from http://irishrail.ie

In [ ]:
df = pd.read_csv("irishrail.csv")
df

From the rail data, we will create an undirected network, with edges for each route. Note that railway routes naturally go in both directions, so an undirected network is appropriate here.

In [ ]:
g = nx.Graph()

In [ ]:
# process each route
for i, row in df.iterrows():
    # add a edge, with "kind" as an attribute
    g.add_edge(row["start"], row["end"], kind=row["kind"])

Check the size of the network:

In [ ]:
print(f"Network has {g.number_of_nodes()} nodes and {g.number_of_edges()} edges")

We can iterate over all of the edges in a network using its `edges()` method. We specify the `data` argument to be True to also return each nodes attribute information:

In [ ]:
for e in g.edges(data=True):
    print(e)

Draw the rail network:

In [ ]:
plt.figure(figsize=(12,10))
nx.draw_networkx(g, 
                 with_labels=True, 
                 node_size=800, 
                 node_color="lightblue")
plt.axis("off")
plt.show()

We could find all routes that service a particular destination using the `neighbors()` method:

In [ ]:
list(g.neighbors("Cork"))

In [ ]:
list(g.neighbors("Limerick"))

## Creating a Network from JSON Data

In our next example, we will read a *JavaScript Object Notation (JSON)* file describing character interactions for the movie *Star Wars: Episode IV*. We will discuss further character networks like this later in the module.

Firstly read in the JSON data:

In [ ]:
import json
# read the raw file
with open("starwars-episode-4.json", "r") as json_file:
    # convert the JSON data into Python data structures
    data = json.load(json_file)

In [ ]:
# check the data
data

Since interactions are naturally reciprocal, we will use an undirected network again to represent the data. Because we have frequency data for the interactions (i.e. the number of times two characters interacted), it makes sense for this to be a weighted undirected network.

Create an empty network and add nodes, where each node represents a different character in the movie. 

Note that we will also create a mapping from the ID numbers in the file to the character names, as we want to use names for node identifiers:

In [ ]:
g = nx.Graph()
name_map = {}
# process all of the characters
for char in data["characters"]:
    name_map[char["id"]] = char["name"]
    g.add_node(char["name"])

In [ ]:
g.number_of_nodes()

Now add the edges based on the interactions in the JSON data:

In [ ]:
# create an edge for each interaction between two characters
for interaction in data["interactions"]:
    name1 = name_map[interaction["id1"]]
    name2 = name_map[interaction["id2"]]
    g.add_edge(name1, name2, weight=interaction["frequency"])

# check the number of edges
g.number_of_edges()

Draw the resulting network:

In [ ]:
plt.figure(figsize=(12, 10))
nx.draw_networkx(g, 
                 with_labels=True, 
                 node_size=800, 
                 node_color="lightblue")
plt.axis("off")
plt.show()

We might want to examine the most frequent interactions in the network to understand which character relationships are most prominent:

In [ ]:
# convert the edges in the network to a Pandas DataFrame
df = nx.to_pandas_edgelist(g)
df.head(10)

In [ ]:
# sort the rows by weight (i.e interaction frequency)
df.sort_values(by="weight", ascending=False).head(10)

We could also plot a histogram showing the distribution of edge weights. As we can see, in most cases the interactions been characters are "once off" events. Once a small number of interactions between specific pairs of chararacters occur frequently.

In [ ]:
ax = df.plot.hist(figsize=(6, 4.5), 
                  fontsize=12, 
                  legend=None, 
                  color="darkred", zorder=3)
ax.set_ylabel("Number of Edges", fontsize=12)
ax.set_xlabel("Edge Weight", fontsize=12)
ax.grid(axis="y")
plt.show()